<a href="https://colab.research.google.com/github/snehalreddythathireddy/SaiSnehalReddy_ldp/blob/search-approaches/search_approaches.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rank-bm25

In [ ]:

import numpy as np
from rank_bm25 import BM25Okapi

# Step 1: Prepare documents
documents = [
    "Artificial Intelligence is transforming modern technology.",
    "Machine Learning enables computers to learn from data.",
    "Deep Learning uses neural networks to solve complex problems.",
    "Python is widely used for Data Science and programming.",
    "SQL is used to manage and query relational databases.",
    "Cricket is a popular sport played in many countries.",
    "Natural Language Processing helps computers understand human language.",
    "Computer Vision enables computers to interpret images and videos."
]

# Step 2: Define user query
query = "How can computers learn from data?"

# Step 3: Tokenize documents
tokenized_docs = [
    doc.lower().split() for doc in documents
]

# Step 4: Create BM25 index
bm25 = BM25Okapi(tokenized_docs)

# Step 5: Tokenize query
tokenized_query = query.lower().split()

# Step 6: Calculate relevance scores
scores = bm25.get_scores(tokenized_query)

# Step 7: Rank documents
ranked_indices = np.argsort(scores)[::-1]

# Step 8: Display top 3 results
print("FULL-TEXT SEARCH RESULTS (BM25)\n")

for idx in ranked_indices[:3]:
    print("Document:", documents[idx])
    print("Score:", round(scores[idx], 4))
    print("-" * 50)


In [ ]:
!pip install -q sentence-transformers

In [ ]:

from sentence_transformers import SentenceTransformer, util
import torch

# Step 1: Prepare documents
documents = [
    "Artificial Intelligence is transforming modern technology.",
    "Machine Learning enables computers to learn from data.",
    "Deep Learning uses neural networks to solve complex problems.",
    "Python is widely used for Data Science and programming.",
    "SQL is used to manage and query relational databases.",
    "Cricket is a popular sport played in many countries.",
    "Natural Language Processing helps computers understand human language.",
    "Computer Vision enables computers to interpret images and videos."
]

# Step 2: Define user query
query = "How can computers learn from data?"

# Step 3: Load Hugging Face embedding model
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Step 4: Generate document embeddings
document_embeddings = model.encode(
    documents,
    convert_to_tensor=True
)

# Step 5: Generate query embedding
query_embedding = model.encode(
    query,
    convert_to_tensor=True
)

# Step 6: Calculate cosine similarity
scores = util.cos_sim(
    query_embedding,
    document_embeddings
)[0]

# Step 7: Rank documents
ranked_indices = torch.argsort(
    scores,
    descending=True
)

# Step 8: Display top 3 results
print("SEMANTIC SEARCH RESULTS (DENSE)\n")

for idx in ranked_indices[:3]:
    idx = idx.item()
    print("Document:", documents[idx])
    print("Similarity:", round(scores[idx].item(), 4))
    print("-" * 50)


In [ ]:
!pip install -q rank-bm25 sentence-transformers

In [ ]:

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

# Step 1: Prepare documents
documents = [
    "Artificial Intelligence is transforming modern technology.",
    "Machine Learning enables computers to learn from data.",
    "Deep Learning uses neural networks to solve complex problems.",
    "Python is widely used for Data Science and programming.",
    "SQL is used to manage and query relational databases.",
    "Cricket is a popular sport played in many countries.",
    "Natural Language Processing helps computers understand human language.",
    "Computer Vision enables computers to interpret images and videos."
]

# Step 2: Define user query
query = "How can computers learn from data?"

# Step 3: BM25 keyword search
tokenized_docs = [
    doc.lower().split() for doc in documents
]

bm25 = BM25Okapi(tokenized_docs)

tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)

# Step 4: Dense vector search
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

document_embeddings = model.encode(
    documents,
    convert_to_tensor=True
)

query_embedding = model.encode(
    query,
    convert_to_tensor=True
)

dense_scores = util.cos_sim(
    query_embedding,
    document_embeddings
)[0].cpu().numpy()

# Step 5: Normalize scores
def normalize(scores):
    scores = np.array(scores)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (
        (scores - scores.min()) /
        (scores.max() - scores.min())
    )

normalized_bm25 = normalize(bm25_scores)
normalized_dense = normalize(dense_scores)

# Step 6: Combine scores
hybrid_scores = (
    0.5 * normalized_bm25 +
    0.5 * normalized_dense
)

# Step 7: Rank documents
ranked_indices = np.argsort(hybrid_scores)[::-1]

# Step 8: Display top 3 results
print("HYBRID SEARCH RESULTS\n")

for idx in ranked_indices[:3]:
    print("Document:", documents[idx])
    print("BM25 Score:", round(normalized_bm25[idx], 4))
    print("Dense Score:", round(normalized_dense[idx], 4))
    print("Hybrid Score:", round(hybrid_scores[idx], 4))
    print("-" * 50)


In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Step 1: Prepare documents
documents = [
    "Artificial Intelligence is transforming modern technology.",
    "Machine Learning enables computers to learn from data.",
    "Deep Learning uses neural networks to solve complex problems.",
    "Python is widely used for Data Science and programming.",
    "SQL is used to manage and query relational databases.",
    "Cricket is a popular sport played in many countries.",
    "Natural Language Processing helps computers understand human language.",
    "Computer Vision enables computers to interpret images and videos."
]

# Step 2: Define user query
query = "How can computers learn from data?"

# Step 3: Load SPLADE model
model_name = "naver/splade-cocondenser-ensembledistil"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForMaskedLM.from_pretrained(model_name)

model.eval()

# Step 4: Generate sparse vector
def generate_sparse_vector(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    weights = torch.log1p(torch.relu(logits))

    weights = weights * inputs["attention_mask"].unsqueeze(-1)

    sparse_vector = torch.max(
        weights,
        dim=1
    ).values[0]

    indices = torch.nonzero(
        sparse_vector,
        as_tuple=True
    )[0]

    values = sparse_vector[indices]

    return indices, values

# Step 5: Generate document sparse vectors
document_vectors = [
    generate_sparse_vector(doc)
    for doc in documents
]

# Step 6: Generate query sparse vector
query_indices, query_values = generate_sparse_vector(query)

# Step 7: Calculate sparse dot-product scores
scores = []

for doc_indices, doc_values in document_vectors:

    common_indices, query_positions, doc_positions = np.intersect1d(
        query_indices.numpy(),
        doc_indices.numpy(),
        return_indices=True
    )

    score = torch.sum(
        query_values[query_positions] *
        doc_values[doc_positions]
    ).item()

    scores.append(score)

# Step 8: Rank documents
ranked_indices = np.argsort(scores)[::-1]

# Step 9: Display top 3 results
print("SPARSE VECTOR SEARCH RESULTS (SPLADE)\n")

for idx in ranked_indices[:3]:
    print("Document:", documents[idx])
    print("Score:", round(scores[idx], 4))
    print("-" * 50)
 explain